
# Cheat Sheet: Discrete Response Models in Python
### Logit / Probit · Multinomial Logit · Poisson · Negative Binomial

A single-page-per-topic quick reference. Copy-paste the snippets and swap in your own
variables/formulas. Pair with `06_Background_Theory.md` for the "why," and `04_Reusable_Template.ipynb`
for a ready-to-run project skeleton.



---
## 0. Decision Tree — "Which model do I use?"

```
Is the outcome variable...
├── Binary (0/1, yes/no)?
│      └── Use LOGIT (interpretable odds ratios) or PROBIT (if you need a latent-normal
│          interpretation, or you're matching a field convention like econometrics of choice).
│
├── Unordered categorical with 3+ levels?
│      └── Use MULTINOMIAL LOGIT (MNLogit / sklearn multinomial LogisticRegression).
│         (If categories are ORDERED, e.g. low/med/high, use Ordered Logit/Probit instead —
│          not covered in this lab, see statsmodels OrderedModel.)
│
└── A count (0, 1, 2, 3, ... with no natural upper bound)?
       ├── Compute mean(y) and var(y).
       ├── var ≈ mean  → POISSON regression.
       └── var > mean (overdispersion) → NEGATIVE BINOMIAL regression.
              (var < mean, underdispersion, is rarer — consider quasi-Poisson or
               generalized Poisson; not covered here.)
```



---
## 1. Logit / Probit (binary outcome)

**Formula:** $\log\frac{P}{1-P} = \beta_0 + \beta_1 x_1 + \dots$ (logit) or
$\Phi^{-1}(P) = \beta_0 + \beta_1 x_1 + \dots$ (probit)


In [ ]:

import statsmodels.formula.api as smf
import numpy as np

# --- Fit ---
logit_model  = smf.logit('y ~ x1 + x2', data=train_df).fit()
probit_model = smf.probit('y ~ x1 + x2', data=train_df).fit()

# --- Inspect ---
logit_model.summary()
logit_model.params            # coefficients (log-odds units)
np.exp(logit_model.params)    # odds ratios (logit only — do NOT do this for probit)
logit_model.prsquared         # McFadden pseudo-R^2

# --- Marginal effects (works for both, and is the fair way to compare logit vs probit) ---
logit_model.get_margeff(at='mean').summary()
probit_model.get_margeff(at='mean').summary()

# --- Predict ---
y_prob = logit_model.predict(test_df[['x1', 'x2']])
y_pred = (y_prob >= 0.5).astype(int)   # adjust threshold as needed


In [ ]:

# --- Evaluate ---
from sklearn.metrics import (confusion_matrix, accuracy_score, ConfusionMatrixDisplay,
                              roc_curve, roc_auc_score, precision_score, recall_score, f1_score)

accuracy_score(y_test, y_pred)
precision_score(y_test, y_pred)
recall_score(y_test, y_pred)
f1_score(y_test, y_pred)
roc_auc_score(y_test, y_prob)

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm).plot()

fpr, tpr, thresh = roc_curve(y_test, y_prob)



**Interpretation cheat lines**
- Logit coefficient $\beta$: "a 1-unit increase in $x$ multiplies the odds of $y=1$ by $e^{\beta}$."
- Probit coefficient: don't interpret directly — use marginal effects: "a 1-unit increase in $x$
  changes $P(y=1)$ by [marginal effect] percentage points, at the mean of other variables."
- `prsquared` (McFadden's pseudo-$R^2$) is NOT comparable to OLS $R^2$. Values of 0.2–0.4 are
  often considered a good fit in this context.

**Gotchas**
- `smf.logit`/`smf.probit` need the dependent variable coded exactly 0/1 (not True/False strings).
- Perfect separation (a predictor perfectly predicts the outcome) causes MLE to fail to converge —
  watch for huge coefficients / huge standard errors.
- Default classification threshold of 0.5 is a choice, not a law — pick it based on the cost of
  false positives vs. false negatives.



---
## 2. Multinomial Logit (3+ unordered classes)

**Formula:** $P(y=k) = \dfrac{e^{z_k}}{\sum_j e^{z_j}}$ (softmax), one $z_k$ equation per non-baseline class.


In [ ]:

import statsmodels.api as sm
import statsmodels.discrete.discrete_model as sm_discrete
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# --- sklearn approach ---
model_sk = LogisticRegression(solver='lbfgs', max_iter=500)  # multinomial is automatic for multi-class y in modern sklearn
model_sk.fit(X_train, y_train)
pred_sk = model_sk.predict(X_test)

# --- statsmodels approach (needs an explicit constant column) ---
X_train_c = sm.add_constant(X_train)
X_test_c  = sm.add_constant(X_test)
model_stat = sm_discrete.MNLogit(y_train, X_train_c).fit(method='bfgs')
model_stat.summary()

y_hat_stat = model_stat.predict(X_test_c)
pred_stat = np.asarray(y_hat_stat).argmax(1)     # pick highest-probability class

# --- relative risk ratios ---
np.exp(model_stat.params)     # interpreted relative to the baseline (usually class 0)

accuracy_score(y_test, pred_sk)
accuracy_score(y_test, pred_stat)



**Interpretation cheat line**
- `MNLogit` coefficients are log-odds **relative to a baseline category** (statsmodels picks the
  first/lowest-coded category by default). Exponentiate for relative risk ratios: "a 1-unit increase
  in $x$ multiplies the odds of being class $k$ (vs. baseline) by $e^{\beta_k}$."

**Gotchas**
- Older sklearn code you find online may pass `multi_class='multinomial'` to `LogisticRegression` —
  this argument was removed in recent sklearn versions because multinomial handling is now automatic
  whenever the target has more than 2 classes and the solver supports it (e.g. `lbfgs`, `newton-cg`).
- Class imbalance can inflate accuracy; check per-class precision/recall or macro-F1.
- statsmodels needs a numeric 0..K-1 encoded target, not strings.



---
## 3. Poisson Regression (counts)

**Formula:** $\ln(\lambda) = \beta_0 + \beta_1 x_1 + \dots \iff \lambda = e^{\beta_0 + \beta_1 x_1 + \dots}$


In [ ]:

import statsmodels.api as sm

X = sm.add_constant(X)                     # ALWAYS add the constant explicitly for sm.Poisson/GLM
poisson_model = sm.Poisson(y, X).fit()      # or: sm.GLM(y, X, family=sm.families.Poisson()).fit()
poisson_model.summary()

np.exp(poisson_model.params)               # multiplicative effect on the expected count

preds = poisson_model.predict(X)

# with an exposure/offset term (e.g., population, time-at-risk, policy-years)
offset_model = sm.Poisson(y, X, offset=np.log(exposure)).fit()

# quick overdispersion check
ratio = poisson_model.deviance / poisson_model.df_resid   # >> 1 suggests overdispersion


In [ ]:

from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

mean_absolute_error(y, preds)
mean_absolute_percentage_error(y, preds)



**Interpretation cheat line**
- "A 1-unit increase in $x$ multiplies the expected count by $e^{\beta}$, holding other variables
  fixed."

**Gotchas**
- Poisson assumes mean = variance. Always check this before trusting the standard errors.
- Use an `offset=np.log(exposure)` whenever observations differ in "opportunity to occur" (different
  time windows, population sizes, area, etc.) — otherwise your rate comparisons are apples-to-oranges.
- MAE/MAPE are easier for stakeholders to read than deviance; report both when possible.



---
## 4. Negative Binomial Regression (overdispersed counts)

**Formula:** same log-linear mean structure as Poisson, but
$\text{Var}(y) = \mu + \alpha \mu^2$ instead of $\text{Var}(y) = \mu$.


In [ ]:

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.families.family import NegativeBinomial

X = sm.add_constant(X)

# Step 1: fit Poisson to get mu_hat (fitted means)
poisson_model = sm.GLM(y_train, X_train, family=sm.families.Poisson()).fit()

# Step 2: auxiliary OLS regression (Cameron & Trivedi) to estimate alpha, NO intercept
df_aux = pd.DataFrame({
    'y_mu_hat': poisson_model.mu,
    'children': y_train.values,       # rename to your target
})
df_aux['y_auxiliary'] = ((df_aux['children'] - df_aux['y_mu_hat'])**2 - df_aux['y_mu_hat']) / df_aux['y_mu_hat']
ols_model = smf.ols('y_auxiliary ~ y_mu_hat - 1', df_aux).fit()
alpha_hat = ols_model.params.iloc[0]
ols_model.summary()          # check the p-value on y_mu_hat -> tests H0: alpha = 0

# Step 3: fit negative binomial GLM using the estimated alpha
nb_model = sm.GLM(y_train, X_train, family=NegativeBinomial(alpha=alpha_hat)).fit()
nb_model.summary()

# Alternative: let statsmodels estimate alpha jointly via MLE (simpler, sometimes less stable)
nb2_model = sm.NegativeBinomial(y_train, X_train).fit()


In [ ]:

from sklearn.metrics import mean_squared_error
def MSE(y_true, y_pred, squared=True):
    val = mean_squared_error(y_true, y_pred)
    return val if squared else np.sqrt(val)

MSE(y_train, poisson_model.predict(X_train), squared=False)
MSE(y_test,  poisson_model.predict(X_test),  squared=False)
MSE(y_train, nb_model.predict(X_train), squared=False)
MSE(y_test,  nb_model.predict(X_test),  squared=False)

poisson_model.aic, poisson_model.bic
nb2_model.aic, nb2_model.bic          # lower AIC/BIC = better fit, penalizing complexity



**Interpretation cheat line**
- Coefficients are read exactly like Poisson coefficients (multiplicative effect on expected count).
  The difference is in the (usually wider, more honest) standard errors.

**Gotchas**
- Negative binomial requires **count data from a fixed number of trials / Bernoulli process**
  conceptually (per the book) — don't reach for it just because data "look overdispersed" without
  understanding why.
- If `alpha_hat` comes out negative or not significant, you probably don't have real overdispersion —
  stick with Poisson.
- `NegativeBinomial(alpha=...)` in `sm.GLM` takes a **fixed, pre-estimated** alpha; `sm.NegativeBinomial`
  estimates alpha itself via MLE — the two can give slightly different coefficient estimates.



---
## 5. Model Comparison Quick Reference

| Metric | Good for | Notes |
|---|---|---|
| Accuracy / Precision / Recall / F1 | Binary & multinomial classifiers | Watch for class imbalance |
| ROC-AUC | Binary classifiers | Threshold-independent |
| McFadden pseudo-$R^2$ | Logit/probit/MNLogit fit | Not comparable to OLS $R^2$ |
| Log-likelihood | Any MLE-fit model | Higher (less negative) is better, same data/sample only |
| AIC / BIC | Comparing nested or non-nested models on same data | Lower is better; BIC penalizes complexity more |
| Deviance / df_resid | Poisson / NegBin dispersion check | >> 1 suggests overdispersion |
| MAE / MAPE / RMSE | Count model prediction accuracy | Easiest to explain to stakeholders |

**General workflow:** always split train/test (or cross-validate) before trusting any of these
numbers — every example in the source book chapter that reports "100% accuracy" is doing so on a
5-row toy test set, which is not a reliable estimate of real-world performance.
